In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
from decimal import Decimal,ROUND_FLOOR

%matplotlib inline
%matplotlib notebook

## 1.Load file.

In [2]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

## 2. Open Water ###

In [4]:
iters = np.shape(date)[0] # total timestep.

### 2.1 Build up using default settings in excel.

#### 2.2.1 Input data preparation

__a.__ r_swds_pr, r_swds_cp, r_swds_op

In [5]:
r_ow_up = [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.290153173,3.075838592,0
,0,0,0,0]
d_ow_gw = [0,-4.21102E-05,0.004440031,0.00432508,0.004232277,0.414138227,1.269644171,0.409532327
,0.63218448,-0.035114928,-0.03202658,-0.032137395,0.175026863,0.187368246
,0.187235509,-0.033502299,-0.032544556,-0.003722519,0.003278472,0.003154041
,0.003063782,0.002973554,0.002883517,0.002793671,0.002704016,0.002614551
,0.002525275,0.002436188,0.00234729,0.002258581,0.000560166,-0.022059166
,-0.022042365,-0.022133186,-0.022223302,-0.02231323,-0.022402968,-0.022492515
,-0.022581873,1.063926463,0.196113043,-0.004444225,0.001252498,0.228155534,-5.36586E-06
,0.000993745,0.000902745,0.231504996,-0.000343094,0.23136175,1.371823058
,1.610664396,1.609600123,1.609676312,1.597866199,1.431661667,1.430143021
,1.430203464,1.430256293,1.430309056]
q_ow_sdws = [0,0,0,0,0,0,0.422282297,0.494957622,0.702412167,0,0,0,0.044801777
,0.194412167,0.194412167,0,0,7.97973E-17,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0.831983598,0.210970609,0,0,0.204308271,0,0,0,0.244282297,0
,0.244282297,1.514282297,5.070282297,7.610282297,12.6902823,4.562282297
,0.227893318,8.32667E-17,0,0,0]
q_ow_mss = np.zeros(iters)
so_ow_sdws = np.zeros(iters)
so_ow_mss = np.zeros(iters)

__b.__ open water level, taken from excel.

#### 2.2.2 Using Arrays to build up the structure first.

In [6]:
# areas for different area
pr_area = 1560
cp_area = 803.3906406
op_area = 481.6093594
up_area = 6855
gw_area = 8140
ow_area = 300
pr_discfrac = 0.5
cp_discfrac, op_discfrac = 0, 0
tot_disc_area = pr_area * pr_discfrac + cp_area * cp_discfrac + op_area * op_discfrac
sdws_area = (pr_area + cp_area + op_area - tot_disc_area) * 1 # 1 is the storm drainage fraction 100%
mss_area = (pr_area + cp_area + op_area - tot_disc_area) * 0 # 0 is the mss drainage fraction 0 %




In [7]:
delta_t = 1 / 24

prec_ow = np.zeros(iters)
e_atm_ow = np.zeros(iters)
r_ow = np.zeros(iters)
d_ow = np.zeros(iters)
q_ow = np.zeros(iters)
so_ow = np.zeros(iters)
r_meas_ow = np.zeros(iters)
q_out_ow = np.zeros(iters)
owl_ow = np.ones(iters) * 1.5
#owl = np.ones(iters) * 1.5


q_out_ow_cap = 200

owl_target = 1.5 # (1500/10000 see in excel)

t = 1 


while t <= iters - 1:
    
    #  Direct rainfall on open water during the current time step [mm].
    if ow_area == 0:
        prec_ow[t] = 0
    else:
        prec_ow[t] = P_atm[t]
        
    # Evarporation from open water during current time step [mm]
    if ow_area == 0:
        e_atm_ow[t] = 0
    else:
        e_atm_ow[t] = E_pot_OW[t]
    
    # Total runoff (from unpaved area) to open water during current time step [mm]
    if ow_area == 0:
        r_ow[t] = 0
    else:
        r_ow[t] = r_ow_up[t] * up_area / ow_area
        
    # Drainage from groundwater to open water during current time step [mm]
    if ow_area ==0:
        d_ow[t] = 0
    else:
        d_ow[t] = d_ow_gw[t] * gw_area / ow_area
        
    # Total outflow from sewer systems to open water during current time step [mm]
    if ow_area ==0:
        q_ow[t] = 0
    else:
        q_ow[t] = (q_ow_sdws[t] * sdws_area + q_ow_mss[t] * mss_area) / ow_area
        
    # Total overflow from sewer systems (to open water) during current time step [mm]
    if ow_area ==0:
        so_ow[t] = 0
    else:
        so_ow[t] = (so_ow_sdws[t] * sdws_area + so_ow_mss[t] * mss_area) / ow_area
        
    # Inflow from measure area (if applicable) during current time step [mm]
    r_meas_ow[t] = 0
    
    # Discharge from open water to outside water during current time step [mm]
    if ow_area ==0:
        q_out_ow[t] = 0
    else:
        q_out_ow[t] = ow_area / 10000 * min( delta_t * q_out_ow_cap *(10000 /ow_area), 1000 * (owl_target - owl_ow[t-1]) + prec_ow[t] - e_atm_ow[t] + r_ow[t] + d_ow[t] + q_ow[t] + so_ow[t] +r_meas_ow[t])

    # Open water level at the end of the current time step [mm]
    if ow_area ==0:
        owl_ow[t] = owl_target
    else:
        owl_ow[t] = owl_ow[t-1] - (prec_ow[t] - e_atm_ow[t] + r_ow[t] + d_ow[t] + q_ow[t] + so_ow[t] + r_meas_ow[t] -(10000 / ow_area) * q_out_ow[t]) /1000
    t += 1
    
#print(r_sdws)
filename = 'Results_openwater_arraybuildup_test.csv'
np.savetxt('sol/' + filename, np.c_[prec_ow, e_atm_ow, r_ow, d_ow, q_ow, so_ow, q_out_ow, owl_ow], fmt = "%.8f", delimiter=',', header = 'prec_ow, e_atm_ow, r_ow, d_ow, q_ow, so_ow, q_out_ow, owl_ow') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

#print('The results have been validated with excel.')

#### 2.2.3 Using Class to build up the module

In [ ]:
class OpenWater:
    def __init__(self, gw_area, seep_def = 0, w = 100, vc = 20000, h_deepgw = 21.5, flux = 1, init_gwl = 1.5, croptype = 2, soiltype = 1):
        
        # state
        self.init_gwl = init_gwl
        
        # parameter
        self.gw_area =  gw_area
        self.seep_def = seep_def
        self.w =  w
        self.vc = vc
        self.h_deepgw = h_deepgw
        self.flux = flux
        self.soiltype = soiltype
        self.croptype = croptype
        self.prev_gwl = init_gwl
        self.prev_gwl_sl = 0
    
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are current precipitation and evaporation.'
    
    def sol(self, p_uz_gw, uz_area, p_op_gw, prev_owl, op_area, delta_t = 1 / 24): 
        
        # sum_p_gw
        sum_p_gw = (p_uz_gw * uz_area + p_op_gw * op_area) / self.gw_area

        # Inflow from measure area (if applicable), set as 0 for the time being
        r_meas_gw = 0
        
        # gwl_up
        c = float(self.prev_gwl)
        if c>= 0.0 and c <= 2.5:
            c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
        elif c < 3.0:
            c = 2.5
        elif c < 5.0:
            c = int(c)
        elif c <= 10:
            c = 5.0
        else:
            c = 10.0
        gwl_up = c
    
        # gwl_low
        if gwl_up < 2.5:
            gwl_low = round(gwl_up + 0.1, 2)
        elif gwl_up < 3:
            gwl_low = 3
        elif gwl_up < 4:
            gwl_low = 4
        elif gwl_up < 5:
            gwl_low = 5
        else:
            gwl_low = 10
        
        # Storage coefficient of the groundwater for the current time step
        if self.prev_gwl < 10:
            sc_gw = SoilSelector(self.soiltype, self.croptype, gwl_low)['stor_coef'].values + (gwl_low - self.prev_gwl) / (gwl_low - gwl_up) * (SoilSelector(self.soiltype, self.croptype, gwl_up)['stor_coef'].values - SoilSelector(self.soiltype, self.croptype, gwl_low)['stor_coef'].values)
        else:
            sc_gw = SoilSelector(self.soiltype, self.croptype, 10)['stor_coef'].values
        
        # Groundwater level at the end of the current time step [m-SL].
        if self.seep_def > 0.5:
            h_gw = -(((sum_p_gw + r_meas_gw) / 1000 * self.w * self.vc - self.h_deepgw * self.w - prev_owl * self.vc) / (self.w + self.vc) + (-(self.prev_gwl + self.prev_gwl_sl) - ((sum_p_gw + r_meas_gw) / 1000 * self.w * self.vc - self.h_deepgw * self.w - prev_owl * self.vc) / (self.w + self.vc)) * np.exp(- delta_t * (self.w + self.vc) /(sc_gw * self.w * self.vc)))
        else:
            h_gw = - (self.w * (((sum_p_gw + r_meas_gw) - self.flux) / 1000) - prev_owl + (-(self.prev_gwl + self.prev_gwl_sl) - (self.w * (((sum_p_gw + r_meas_gw)- self.flux) / 1000) - prev_owl)) * np.exp(- delta_t / (sc_gw * self.w )))         
       
        # Downward seepage flux to deep groundwater during current time step.
        if self.seep_def < 0.5:
            s_out = delta_t * self.flux
        else:
            s_out = 1000 * (self.h_deepgw - 0.5 * (h_gw + (self.prev_gwl + self.prev_gwl_sl))) / self.vc * delta_t
        
        # Groundwater drainage to the open water for the current time step [mm].
        d_ow = sum_p_gw + r_meas_gw - s_out - sc_gw * (self.prev_gwl + self.prev_gwl_sl -  h_gw) * 1000        
 
        # Groundwater level below surface level at the end of the current time step [m-SL].
        gwl = max(0, self.prev_gwl - (sum_p_gw + r_meas_gw - s_out - d_ow) / (1000 * sc_gw))
    
        # Groundwater level above surface level at the end of the current time step [m-SL]
        gwl_sl = -1 * max(0, (0 - (self.prev_gwl - (sum_p_gw + r_meas_gw - s_out - d_ow)/(1000 * sc_gw))) * sc_gw)          
        
            
        # update state
        self.prev_gwl = gwl
        self.prev_gwl_sl = gwl_sl
         
        return sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl

In [ ]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(2, 1, 1.5)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.5]
gwl_sl= [0]

# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 0, w = 100, vc = 20000, h_deepgw = 21.5, flux = 1, init_gwl = 1.5, croptype = 2, soiltype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_sewersystem_c1s1.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

### 2.3 Validate with excel with two sets of coefficients.

#### 2.3.1 C2S1

Drainage resistance 80; 
seepage = 0;
flux = 2;
init_gwl = 1.6;
h_deepgw = 23;
flow resistence  = 30000;

__Note that__ when you change the __draiange resistance, flux and init_gwl__, the gwl is automatically changed, __so the input percolation from unsaturated zone is also changed.

In [ ]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(2, 1, 1.6)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.6]
gwl_sl= [0]


# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 0, w = 80, vc = 30000, h_deepgw = 23, flux = 2, init_gwl = 1.6, croptype = 2, soiltype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_Groundwater_c2s1.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

#### 2.3.2 C2S2

Drainage resistance 120; 
seepage = 1;
flux = 1.5;
init_gwl = 1.3;
h_deepgw = 20;
flow resistence  = 25000;
Soiltype = 3 
croptype = 1

In [ ]:
p_uz_gw =[0,0,0.012454488,0.01235691,0.012314248,0.520271547,1.577586236,0.512277858,0.787725029,-0.036021096,-0.032474091,-0.032532207
,0.221298711,0.236757085,0.236652687,-0.03386665,-0.032724943,0.0030632,0.011826368,0.011748062,0.01170814,0.011668236,0.011628513
,0.011588971,0.011549608,0.011510423,0.011471415,0.011432582,0.011393924,0.011355439,0.009322423,-0.018633047,-0.01855179,-0.018590603
,-0.01862873,-0.018666688,-0.018704478,-0.0187421,-0.018779555,1.32458118,0.249958166,0.003909225,0.010918914,0.289205683,0.009622427
,0.010789775,0.010748858,0.293614775,0.009466674,0.293553116,1.706810486,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875
,1.805383626]
p_op_gw = [0,0,0,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0.041666667,0.041666667,0,0,0.041666667,0,0,0,0.041666667,0,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667
,0.041666667,0,0,0,0]

In [ ]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(3, 1, 1.3)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.3]
gwl_sl= [0]


# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 1, w = 120, vc = 25000, h_deepgw = 20, flux = 1.5, init_gwl = 1.3, soiltype = 3, croptype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_Groundwater_c2s2.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')